# Task B -- full fit on every training row

One model, all 3,143 rows, no folds and no held-out set. About 25 minutes of
domain adaptation plus 20 minutes of training, against 90 minutes for the
five-fold version.

**Read this before you submit the result.** This run has no score and cannot
have one. Every labelled row went into training, so there is nothing left to
measure it against. The only way to compare it with the five-fold run's 0.6013
is to submit both and read the leaderboard, and the leaderboard carries about
three points of noise on 395 rows, so a difference smaller than that tells you
nothing either way.

The one comparison that does exist points the other way. `b_tapt` trained as a
single model on 85% of the rows and scored 0.5972. The five-fold average of
models each seeing 80% scored 0.6013. Averaging beat the extra data.

Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on. Save & Run All.


In [ ]:
import os, subprocess, sys, pathlib
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","task-b","--depth","1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK); print("cwd:", os.getcwd())
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print(f"$ {' '.join(cmd)}", flush=True)
    fh = open(log,"w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Domain-adapt MuRIL (~25 min)

Same step that carried the whole gain last time. Skipped automatically if the
checkpoint is already present.


In [ ]:
if not os.path.isdir("work/runs/tapt-muril"):
    run([sys.executable,"-u","work/tapt.py","--out","work/runs/tapt-muril"],
        log="work/tapt.log")
else:
    print("tapt-muril already present, skipping")


## 2. Fit every row (~20 min)

`--folds 1` is the new mode. It trains on all 3,143 rows, keeps the final
checkpoint because there is no validation set to select one with, and prints the
predicted class distribution against the training prior. That distribution is
the only diagnostic available here, so read it: a model that has collapsed onto
`Gender` will show it there.


In [ ]:
run([sys.executable,"-u","work/muril_b.py","--tag","b_full",
     "--model","work/runs/tapt-muril","--folds","1","--seeds","42","--epochs","6"],
    log="work/b_full.log")


## 3. Package


In [ ]:
ZIP = "/kaggle/working/b_full.zip"
run([sys.executable,"work/make_submission.py","--task","b",
     "--pred","work/runs/b_full/predictions.csv","--out",ZIP])
run(["cp","work/b_full.log","/kaggle/working/"])
assert os.path.exists(ZIP)
run(["unzip","-l",ZIP])
print("\nDownload b_full.zip from the Output tab.")
print("It has no score. If it beats 0.5922 on CodaBench by less than ~3 points,")
print("that is noise and not evidence that full-fitting helped.")
